In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torchsummary import summary

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
if device == 'cuda':
    torch.cuda.manual_seed_all(42)
print(device + " is available")

cuda is available


In [3]:
train_dataset = datasets.FashionMNIST(root='./data',  # 데이터 저장 위치
                                      train=True,     # true이면 train dataset 다운, false이면 test dataset 다운
                                      transform=transforms.ToTensor(), # 모든 픽셀 값을 0~1사이로 만듬
                                      download=True)
test_dataset = datasets.FashionMNIST(root='./data',
                                     train=False,
                                     transform=transforms.ToTensor())

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 171kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.18MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 24.0MB/s]


In [4]:
len(train_dataset)

60000

In [ ]:
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=64,
                                           shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=64,
                                          shuffle=False)

In [ ]:
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.fc1 = nn.Linear(784, 512) # input: 784, output: 512
    self.fc2 = nn.Linear(512, 256) # input: 512, output: 256
    self.fc3 = nn.Linear(256, 128) # input: 256, output: 128
    self.fc4 = nn.Linear(128, 64)  # input: 128, output: 64
    self.fc5 = nn.Linear(64, 10)   # input: 64, output: 10

  def forward(self, x):
    x = x.view(-1, 784)  # vectorize
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.relu(self.fc3(x))
    x = F.relu(self.fc4(x))
    x = self.fc5(x)
    return x

In [ ]:
# sequential 사용
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.layer1 = nn.Sequential(
        nn.Linear(784, 512),
        nn.ReLU(inplace=True)
    )
    self.layer2 = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(inplace=True)
    )
    self.layer3 = nn.Sequential(
        nn.Linear(256, 128),
        nn.ReLU(inplace=True)
    )
    self.layer4 = nn.Sequential(
        nn.Linear(128, 64),
        nn.ReLU(inplace=True)
    )
    self.layer5 = nn.Sequential(
        nn.Linear(64, 10)
    )

  def forward(self, x):
    x = x.view(-1, 784)  # vectorize
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.layer5(x)
    return x

In [ ]:
# Instantiate the DNN model, loss function, and optimizer
model = Net().to(device)
summary(model, (1, 28, 28))
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                  [-1, 512]         401,920
              ReLU-2                  [-1, 512]               0
            Linear-3                  [-1, 256]         131,328
              ReLU-4                  [-1, 256]               0
            Linear-5                  [-1, 128]          32,896
              ReLU-6                  [-1, 128]               0
            Linear-7                   [-1, 64]           8,256
              ReLU-8                   [-1, 64]               0
            Linear-9                   [-1, 10]             650
Total params: 575,050
Trainable params: 575,050
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 2.19
Estimated Total Size (MB): 2.21
-------------------------------------------

In [ ]:
num_epochs = 10
for epoch in range(num_epochs):
  for batch_idx, (data, target) in  enumerate(train_loader):
    data = data.to(device)
    target = target.to(device)

    optimizer.zero_grad()                # 모든 model의 gradient 값을 0으로 설정
    hypothesis = model(data)             # 모델을 forward pass해 결과값 저장
    cost = criterion(hypothesis, target) # output과 target의 loss 계산
    cost.backward()                      # backward 함수를 호출해 gradient 계산
    optimizer.step()                     # 모델의 학습 파라미터 갱신

    if batch_idx % 100 == 0:
      print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx}/{len(train_loader)}], Loss: {cost.item()}')

Epoch [1/10], Batch [0/938], Loss: 2.29069185256958
Epoch [1/10], Batch [100/938], Loss: 2.3010973930358887
Epoch [1/10], Batch [200/938], Loss: 2.2908191680908203
Epoch [1/10], Batch [300/938], Loss: 2.2919235229492188
Epoch [1/10], Batch [400/938], Loss: 2.2947773933410645
Epoch [1/10], Batch [500/938], Loss: 2.280205726623535
Epoch [1/10], Batch [600/938], Loss: 2.262784481048584
Epoch [1/10], Batch [700/938], Loss: 2.240941286087036
Epoch [1/10], Batch [800/938], Loss: 2.181217670440674
Epoch [1/10], Batch [900/938], Loss: 2.101034164428711
Epoch [2/10], Batch [0/938], Loss: 1.9958093166351318
Epoch [2/10], Batch [100/938], Loss: 1.6908140182495117
Epoch [2/10], Batch [200/938], Loss: 1.4375674724578857
Epoch [2/10], Batch [300/938], Loss: 1.2740248441696167
Epoch [2/10], Batch [400/938], Loss: 1.1136622428894043
Epoch [2/10], Batch [500/938], Loss: 1.2591112852096558
Epoch [2/10], Batch [600/938], Loss: 1.0604043006896973
Epoch [2/10], Batch [700/938], Loss: 1.1105377674102783
Epo

In [ ]:
model.eval()           # evaluate mode로 전환 dropout 이나 batch_normalization 해제
with torch.no_grad():  # grad 해제
    correct = 0
    total = 0
    for data, target in test_loader:
        data = data.to(device)
        target = target.to(device)
        output = model(data)
        _, predicted = torch.max(output.data, axis=1)      # 출력이 분류 각각에 대한 값으로 나타나기 때문에, 가장 높은 값을 갖는 인덱스를 추출
        total += len(target)                          # 전체 클래스 개수
        correct += (predicted == target).sum().item() # 예측값과 실제값이 같은지 비교
    accuracy = 100 * correct / total
    print(f'Test Accuracy: {accuracy}%')

Test Accuracy: 82.81%
